In [72]:
## 1. Feature Engineering for Cross-Sectional Equity Modeling:

## Objective: Transform cleaned price data into economically meaningful features capturing momentum, volatility, and trend dynamics for downstream modeling and portfolio construction.
## Research Motivation: Raw price series are not directly suitable for predictive modeling. In quantitative finance, feature engineering is used to extract signals that reflect market behavior such as persistence (momentum), dispersion (volatility), and trend deviations.
#This notebook constructs a feature set designed to serve as inputs for systematic trading strategies.

In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

In [74]:
## 2. Set project paths Load preprocessed data
project_root = Path.cwd().parent
processed_data_path = project_root / "data" / "processed"
clean_close_prices = pd.read_csv(processed_data_path/ "clean_close_prices.csv", index_col=0, parse_dates=True)
clean_close_prices.head()

,AAPL,ABBV,ACN,ADBE,AMZN,AVGO,BAC,BRK-B,COST,CRM,...,NFLX,NVDA,PEP,PG,SPY,TMO,UNH,V,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,40.304180,69.343712,136.707809,177.699997,59.450500,21.086666,24.637491,197.220001,168.498230,103.156639,...,20.107000,4.928267,91.602791,72.824989,236.562164,188.755341,193.090408,108.111900,28.855202,58.580429
2018-01-03,40.297153,70.428856,137.338745,181.039993,60.209999,21.317266,24.555101,199.789993,170.520233,104.026054,...,20.504999,5.252614,91.362274,72.736626,238.058441,192.217880,195.115997,109.188194,29.106909,59.730930
2018-01-04,40.484329,70.027206,138.964966,183.220001,60.479500,21.324373,24.876455,200.690002,169.196075,105.399368,...,20.563000,5.280303,91.812286,73.250763,239.061829,194.624008,195.962860,109.594177,29.133244,59.813622
2018-01-05,40.945255,71.246239,140.111282,185.339996,61.457001,21.450724,24.991814,201.419998,167.988144,106.802322,...,20.999001,5.325050,92.076118,73.298973,240.654922,197.959351,199.699692,112.218857,29.305923,59.765404
2018-01-08,40.793175,70.104721,141.230972,185.039993,62.343498,21.502064,24.818779,202.740005,168.641357,107.553200,...,21.205000,5.488213,91.548485,73.684608,241.094971,198.282104,196.233536,112.672050,29.739086,60.034081


In [75]:
## 3. Return Computation
#Returns are the primary variable of interest in financial modeling. I compute simple daily returns as the basis for all subsequent features.

In [76]:
returns = clean_close_prices.pct_change()
returns.head()

,AAPL,ABBV,ACN,ADBE,AMZN,AVGO,BAC,BRK-B,COST,CRM,...,NFLX,NVDA,PEP,PG,SPY,TMO,UNH,V,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2018-01-03,-0.000174,0.015649,0.004615,0.018796,0.012775,0.010936,-0.003344,0.013031,0.012000,0.008428,...,0.019794,0.065814,-0.002626,-0.001213,0.006325,0.018344,0.010490,0.009955,0.008723,0.019640
2018-01-04,0.004645,-0.005703,0.011841,0.012042,0.004476,0.000333,0.013087,0.004505,-0.007765,0.013202,...,0.002829,0.005271,0.004926,0.007068,0.004215,0.012518,0.004340,0.003718,0.000905,0.001384
2018-01-05,0.011385,0.017408,0.008249,0.011571,0.016163,0.005925,0.004637,0.003637,-0.007139,0.013311,...,0.021203,0.008474,0.002874,0.000658,0.006664,0.017137,0.019069,0.023949,0.005927,-0.000806
2018-01-08,-0.003714,-0.016022,0.007991,-0.001619,0.014425,0.002393,-0.006924,0.006554,0.003888,0.007031,...,0.009810,0.030641,-0.005730,0.005261,0.001829,0.001630,-0.017357,0.004038,0.014781,0.004496


In [77]:
## 4. Momentum Features
# Momentum captures trend persistence in asset prices. Assets that performed well in the recent past tend to continue performing well in the short term.
momentum_5 = clean_close_prices / clean_close_prices.shift(5) -1
momentum_20 = clean_close_prices / clean_close_prices.shift(20) -1
momentum_60 = clean_close_prices / clean_close_prices.shift(60) -1

In [78]:
## 5. Volatility Features
# Volatility measures risk and uncertainty. Financial markets exhibit volatility clustering, where periods of high volatility tend to follow high volatility.
volatility_20 = returns.rolling(20).std()
volatility_60 = returns.rolling(60).std()

In [79]:
## 6. Trend Features and signal detection
moving_avg_20 = clean_close_prices.rolling(20).mean()
moving_avg_50 = clean_close_prices.rolling(50).mean()
Fast_signal = clean_close_prices / moving_avg_20 - 1
Slow_signal = clean_close_prices / moving_avg_50 - 1
trend_signal = (moving_avg_20 > moving_avg_50).astype(int)

In [80]:
## 7. Lag features to capture short-term market memory
lag_1 = returns.shift(1)
lag_2 = returns.shift(2)
lag_5 = returns.shift(5)

In [81]:
## 8. Combine Features
returns_feat = returns.add_suffix("_ret")
momentum_5_feat = momentum_5.add_suffix("_mom5")
momentum_20_feat = momentum_20.add_suffix("_mom20")
momentum_60_feat = momentum_60.add_suffix("_mom60")
volatility_20_feat = volatility_20.add_suffix("_vol20")
volatility_60_feat = volatility_60.add_suffix("_vol60")
trend_signal_feat = trend_signal.add_suffix("_trend")
lag_1_feat = lag_1.add_suffix("_lag1")
lag_2_feat = lag_2.add_suffix("_lag2")
lag_5_feat = lag_5.add_suffix("_lag5")
combined_features = pd.concat([returns_feat, momentum_5_feat, momentum_20_feat, momentum_60_feat, volatility_20_feat, volatility_60_feat, trend_signal_feat, lag_1_feat, lag_2_feat, lag_5_feat], axis=1)
combined_features = combined_features.dropna()
combined_features.head()

,AAPL_ret,ABBV_ret,ACN_ret,ADBE_ret,AMZN_ret,AVGO_ret,BAC_ret,BRK-B_ret,COST_ret,CRM_ret,...,NFLX_lag5,NVDA_lag5,PEP_lag5,PG_lag5,SPY_lag5,TMO_lag5,UNH_lag5,V_lag5,WMT_lag5,XOM_lag5
Date,,,,,,,,,,,,,,,,,,,,,
2018-03-29,0.007809,0.004031,0.041313,0.016656,0.011122,-0.004352,0.020415,0.014288,0.026251,0.030298,...,-0.030902,-0.026996,-0.007357,-0.008177,-0.024997,-0.026437,-0.034153,-0.026213,-0.011794,-0.020522
2018-04-02,-0.006556,-0.033703,-0.035961,-0.017586,-0.052061,-0.033821,-0.022674,-0.022458,-0.030144,-0.008598,...,-0.018781,-0.036717,-0.016584,-0.006544,-0.021315,-0.012926,-0.012360,-0.024919,-0.019738,-0.008299
2018-04-03,0.010259,-0.009512,0.005474,0.017194,0.014621,0.039968,0.009553,0.015180,-0.000766,0.010321,...,0.064498,0.049405,0.006218,0.006587,0.027359,0.024579,0.030675,0.031111,0.024350,0.015228
2018-04-04,0.019122,0.025941,0.007931,0.042236,0.013304,0.000887,0.009801,0.010861,0.015936,0.025668,...,-0.061370,-0.077552,0.007771,0.018061,-0.017012,-0.014594,-0.005067,-0.026857,-0.016572,-0.004054
2018-04-05,0.006935,-0.007854,0.010669,-0.006710,0.029194,-0.002869,0.014726,0.003698,0.003935,-0.003683,...,-0.049619,-0.018491,0.014307,0.013498,-0.002955,-0.003388,0.002478,-0.003493,0.019988,-0.012076


In [82]:
## 8. Feature Engineering Conclusion:

# In this notebook, I engineered a set of features from price data, including momentum, volatility, trend, and lagged returns, to capture different aspects of market behavior. 
# I then combined these features into a clean, well-structured dataset with consistent naming and no missing values. 
# This feature matrix provides a solid foundation for building predictive models and trading strategies in the next stage.

In [83]:
combined_features_path = project_root / "data" / "processed"
combined_features_path.mkdir(parents=True, exist_ok=True)
combined_features.to_csv(combined_features_path / "combined_features.csv", index=True)